# 04 — Final predictions

Selected specification from the mature temporal validation in `03_modeling.ipynb`:

**CatBoost core without `start_year` and without `selected_options_value`.**

| Model | Validation MAE (Jan–Jun 2024) |
|---|---|
| CatBoost core (no `start_year`) | 33.20 days |
| Floor-plan-average baseline | 36.81 days |
| Global-average baseline | 50.73 days |

`start_year` is excluded because adding it **worsened** that validation MAE (35.49).  
`selected_options_value` is excluded even though it improved MAE (31.71), because its availability on the construction-start date has not been confirmed.

This notebook trains once on all mature completed starts (2021-01-01 through 2024-06-30) with a **fixed** 159 trees (the validation run’s tree count). No early stopping. No `predictions.csv` rewrite of the input extracts.


In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd()
TRAIN_PATH = ROOT / "data" / "processed" / "build_history_train.csv"
PRED_PATH = ROOT / "data" / "processed" / "homes_to_predict_cleaned.csv"
OUT_PATH = ROOT / "predictions.csv"

assert TRAIN_PATH.exists(), f"Missing training file: {TRAIN_PATH}"
if not PRED_PATH.exists():
    raise FileNotFoundError(
        f"Missing cleaned prediction file: {PRED_PATH}. "
        "Do not fall back to data/homes_to_predict.csv. Run 02_data_cleaning.ipynb first."
    )


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


HASH_BEFORE = {p.name: file_sha256(p) for p in (TRAIN_PATH, PRED_PATH)}
print("train:", TRAIN_PATH)
print("predict:", PRED_PATH)
print("sha256 train ", HASH_BEFORE[TRAIN_PATH.name])
print("sha256 predict", HASH_BEFORE[PRED_PATH.name])


train: C:\Users\zorai\Desktop\construction-time-prediction\data\processed\build_history_train.csv
predict: C:\Users\zorai\Desktop\construction-time-prediction\data\processed\homes_to_predict_cleaned.csv
sha256 train  0dc0c22b16fabecefa4523abd8ea9df352165801ee8e94fdc765659421e1ec20
sha256 predict 00ece846659cdf6a20bf0f160b3a5c0f9bccbce3f0574d42a7eb85e253eac756


## 1. Final training cohort

Eligible completed homes with `construction_start_date` from **2021-01-01 through 2024-06-30** inclusive. This is the 03 training cohort plus the January–June 2024 validation cohort. Starts after 2024-06-30 stay out (completion-selection bias).


In [2]:
train_all = pd.read_csv(TRAIN_PATH)
predict_in = pd.read_csv(PRED_PATH)

train_all["construction_start_date"] = pd.to_datetime(
    train_all["construction_start_date"], errors="coerce"
)
TRAIN_START = pd.Timestamp("2021-01-01")
TRAIN_END = pd.Timestamp("2024-06-30")

final_train = train_all[
    (train_all["construction_start_date"] >= TRAIN_START)
    & (train_all["construction_start_date"] <= TRAIN_END)
].copy()

print("final training rows:", len(final_train))
print("start min / max:", final_train["construction_start_date"].min().date(),
      final_train["construction_start_date"].max().date())
print("prediction rows:", len(predict_in))

assert (final_train["construction_start_date"] >= TRAIN_START).all()
assert (final_train["construction_start_date"] <= TRAIN_END).all()
assert not (final_train["construction_start_date"] > TRAIN_END).any()
assert final_train["cycle_days"].notna().all()
assert (final_train["cycle_days"] > 0).all()
assert len(predict_in) == 578, f"expected 578 prediction homes, got {len(predict_in)}"
print("cohort assertions passed")


final training rows: 3369
start min / max: 2021-01-04 2024-06-28
prediction rows: 578
cohort assertions passed


## 2. Feature preparation (same as selected core model)

One function is applied to both the final training frame and `homes_to_predict_cleaned.csv`. Numeric missings stay `NaN`. Categorical missings become `"Unknown"` strings. `start_year` is not created. `selected_options_value` is not used.


In [3]:
CAT_FEATURES = [
    "community", "metro", "permit_authority", "plan_code", "product_line",
    "basement_type", "lot_type", "site_manager", "is_spec_home",
]
NUM_FEATURES = [
    "beds", "baths", "half_baths", "sqft", "stories", "garage_size", "lot_size_sqft",
]
DERIVED_NUM = [
    "start_month", "start_quarter",
    "permit_processing_days", "permit_to_start_days",
    "sale_recorded_before_start", "days_sale_to_start",
]
FEATURE_ORDER = CAT_FEATURES + NUM_FEATURES + DERIVED_NUM

DATE_COLS = [
    "sale_date", "permit_application_date", "permit_issue_date", "construction_start_date",
]
REQUIRED_INPUT = CAT_FEATURES + NUM_FEATURES + DATE_COLS

EXCLUDED = {
    "home_id", "lot_number", "plan_name",
    "sale_date", "permit_application_date", "permit_issue_date", "construction_start_date",
    "construction_end_date", "final_inspection_date",
    "change_order_count", "trade_invoice_total", "construction_status", "data_source",
    "train_eligible", "exclude_reason", "cycle_days",
    "start_year", "selected_options_value",
}


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    # Row-wise start-time features. Does not fit on one frame and transform another.
    missing = [c for c in REQUIRED_INPUT if c not in df.columns]
    assert not missing, f"missing required input columns: {missing}"

    out = df.copy()
    for c in DATE_COLS:
        out[c] = pd.to_datetime(out[c], errors="coerce")

    out["start_month"] = out["construction_start_date"].dt.month.astype("int64")
    out["start_quarter"] = out["construction_start_date"].dt.quarter.astype("int64")
    out["permit_processing_days"] = (
        out["permit_issue_date"] - out["permit_application_date"]
    ).dt.days
    out["permit_to_start_days"] = (
        out["construction_start_date"] - out["permit_issue_date"]
    ).dt.days

    sale_ok = out["sale_date"].notna() & (out["sale_date"] <= out["construction_start_date"])
    out["sale_recorded_before_start"] = sale_ok.astype("int64")
    out["days_sale_to_start"] = np.where(
        sale_ok, (out["construction_start_date"] - out["sale_date"]).dt.days, np.nan
    )

    out["is_spec_home"] = out["is_spec_home"].map({True: "True", False: "False"}).fillna(
        out["is_spec_home"].astype(str)
    )
    for c in CAT_FEATURES:
        out[c] = out[c].replace({"": np.nan}).where(out[c].notna(), "Unknown").astype(str)
        out.loc[out[c].str.lower().isin(["nan", "none", "<na>"]), c] = "Unknown"

    X = out.loc[:, FEATURE_ORDER].copy()
    for c in CAT_FEATURES:
        X[c] = X[c].astype(str)
    return X


ids_pred = predict_in["home_id"].copy()
X_train = prepare_features(final_train)
y_train = final_train["cycle_days"].astype(float)
X_pred = prepare_features(predict_in)

assert list(X_train.columns) == FEATURE_ORDER
assert list(X_pred.columns) == FEATURE_ORDER
assert list(X_train.columns) == list(X_pred.columns)
assert "start_year" not in X_train.columns and "start_year" not in X_pred.columns
assert "selected_options_value" not in X_train.columns
assert "selected_options_value" not in X_pred.columns
assert set(X_train.columns).isdisjoint(EXCLUDED)
assert set(X_pred.columns).isdisjoint(EXCLUDED)
assert "home_id" not in X_train.columns and "home_id" not in X_pred.columns
assert ids_pred.notna().all() and ids_pred.is_unique
assert len(ids_pred) == len(X_pred) == 578
assert len(X_train) == len(final_train) == len(y_train)

print("X_train", X_train.shape, "| X_pred", X_pred.shape)
print("feature order:", FEATURE_ORDER)


X_train (3369, 22) | X_pred (578, 22)
feature order: ['community', 'metro', 'permit_authority', 'plan_code', 'product_line', 'basement_type', 'lot_type', 'site_manager', 'is_spec_home', 'beds', 'baths', 'half_baths', 'sqft', 'stories', 'garage_size', 'lot_size_sqft', 'start_month', 'start_quarter', 'permit_processing_days', 'permit_to_start_days', 'sale_recorded_before_start', 'days_sale_to_start']


## 3. Final CatBoost fit

Same hyperparameters as `03_modeling.ipynb`, except `iterations=159` (the selected core model’s tree count) and **no** evaluation set / early stopping, because Jan–Jun 2024 rows are now in the training cohort.


In [4]:
CATBOOST_PARAMS = dict(
    loss_function="RMSE",
    eval_metric="MAE",
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_SEED,
    iterations=159,
    verbose=False,
    allow_writing_files=False,
)

train_pool = Pool(X_train, y_train, cat_features=CAT_FEATURES)
pred_pool = Pool(X_pred, cat_features=CAT_FEATURES)

model = CatBoostRegressor(**CATBOOST_PARAMS)
model.fit(train_pool)
print("fitted trees:", model.tree_count_)
assert int(model.tree_count_) == 159


fitted trees: 159


## 4. Predictions → project-root `predictions.csv`

Raw model outputs must be finite and positive. They are then rounded to the nearest whole day and stored as integers. No clipping or manual replacement.


In [5]:
raw_pred = np.asarray(model.predict(pred_pool), dtype=float)
assert len(raw_pred) == 578
assert np.isfinite(raw_pred).all(), "non-finite raw predictions"
assert (raw_pred > 0).all(), "non-positive raw predictions"

rounded = np.rint(raw_pred)
assert np.isfinite(rounded).all()
assert (rounded > 0).all(), "rounding produced a non-positive day count"

predictions = pd.DataFrame({
    "home_id": ids_pred.to_numpy(),
    "predicted_cycle_days": rounded.astype("int64"),
})
assert list(predictions.columns) == ["home_id", "predicted_cycle_days"]
assert len(predictions) == 578
assert predictions["home_id"].is_unique
assert predictions["home_id"].notna().all()
assert (predictions["home_id"].to_numpy() == predict_in["home_id"].to_numpy()).all()
assert pd.api.types.is_integer_dtype(predictions["predicted_cycle_days"])
assert (predictions["predicted_cycle_days"] > 0).all()

# Write only to the project root.
assert OUT_PATH.parent.resolve() == ROOT.resolve()
assert "data" not in OUT_PATH.parts[-2:]
predictions.to_csv(OUT_PATH, index=False)
print("wrote", OUT_PATH)


wrote C:\Users\zorai\Desktop\construction-time-prediction\predictions.csv


## 5. Re-read and verify `predictions.csv`


In [6]:
assert OUT_PATH.resolve() == (ROOT / "predictions.csv").resolve()
reloaded = pd.read_csv(OUT_PATH)

assert list(reloaded.columns) == ["home_id", "predicted_cycle_days"]
assert len(reloaded) == 578
assert reloaded["home_id"].is_unique
assert reloaded["home_id"].notna().all()
assert (reloaded["home_id"].to_numpy() == predict_in["home_id"].to_numpy()).all()
assert reloaded["predicted_cycle_days"].notna().all()
assert np.isfinite(reloaded["predicted_cycle_days"].to_numpy(dtype=float)).all()
assert (reloaded["predicted_cycle_days"] > 0).all()
assert pd.api.types.is_integer_dtype(reloaded["predicted_cycle_days"]) or (
    (reloaded["predicted_cycle_days"] % 1 == 0).all()
)

print("first 10 rows:")
display(reloaded.head(10))
print("rows:", len(reloaded), "| unique home_id:", reloaded["home_id"].nunique())
print(
    "predicted_cycle_days min/mean/median/max:",
    int(reloaded["predicted_cycle_days"].min()),
    float(reloaded["predicted_cycle_days"].mean()),
    float(reloaded["predicted_cycle_days"].median()),
    int(reloaded["predicted_cycle_days"].max()),
)
print("output path:", OUT_PATH.resolve())

hash_after = {p.name: file_sha256(p) for p in (TRAIN_PATH, PRED_PATH)}
print("inputs unchanged:", hash_after == HASH_BEFORE)
assert hash_after == HASH_BEFORE, "Training or prediction input CSV was modified."
print("done")


first 10 rows:


,home_id,predicted_cycle_days
0,H105165,140
1,H105228,167
2,H104895,162
3,H105114,155
4,H105155,234
5,H104917,260
6,H105019,204
7,H104800,207
8,H105143,152
9,H105002,200


rows: 578 | unique home_id: 578
predicted_cycle_days min/mean/median/max: 105 184.3477508650519 173.5 355
output path: C:\Users\zorai\Desktop\construction-time-prediction\predictions.csv
inputs unchanged: True
done
